In [1]:
# Import necessary libraries for file handling, data manipulation, and visualization
import os
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Import libraries for working with images and transformations
from PIL import Image
import cv2 as cv

# Import PyTorch modules for model building, data handling, and evaluation
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torch.nn.functional as F
import torchvision.models as models
import torchvision.models.quantization as quant_models
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader, Subset
from timm import create_model

# from torchinfo import summary

# Import libraries for machine learning metrics and model evaluation
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, r2_score, confusion_matrix
# import torchmetric
from tqdm import tqdm
from datetime import datetime
import json
import csv

import warnings
warnings.filterwarnings('ignore')
import gc

# Set the seed.
seed = 42
torch.manual_seed(seed)

/home/sebastian-cruz6/cp-anemia-detection/cawt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [32]:
data_dir="/home/sebastian-cruz6/cp-anemia-detection/data/cp-anemia/"
weights_dir="/home/sebastian-cruz6/cp-anemia-detection/notebooks/weights/"
metrics_dir="/home/sebastian-cruz6/cp-anemia-detection/notebooks/metrics/"

# data_dir = "/content/drive/MyDrive/CAWT_Sebastian_202425/CP-AnemiC/"
# weights_dir = "/content/drive/MyDrive/CAWT_Sebastian_202425/Weights/"
anemic_dir=data_dir+"/Anemic/"
non_anemic_dir=data_dir+"/Non-anemic/"
signature = "022505"

In [3]:
data_sheet_path = data_dir+"Anemia_Data_Collection_Sheet.csv"
data_sheet = pd.read_csv(data_sheet_path)
display(data_sheet)

,IMAGE_ID,HB_LEVEL,Severity,Age(Months),GENDER,REMARK,HOSPITAL,CITY/TOWN,MUNICIPALITY/DISTRICT,REGION,COUNTRY
0,Image_001,9.80,Moderate,6,Female,Anemic,Nkawie-Toase Government Hospital,Nkawie-Toase,Atwima Nwabiagya South,Ashanti,Ghana
1,Image_002,9.90,Moderate,24,Male,Anemic,Ejusu Government Hospital,Ejusu,Ejusu Municipality,Ashanti,Ghana
2,Image_003,11.10,Non-Anemic,24,Female,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
3,Image_004,12.50,Non-Anemic,12,Male,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
4,Image_005,9.90,Moderate,24,Male,Anemic,Sunyani Municipal Hospital,Sunyani,Sunyani Municipality,Bono,Ghana
...,...,...,...,...,...,...,...,...,...,...,...
705,Image_706,12.80,Non-Anemic,48,Male,Non-anemic,Bolgatanga Regional Hospital,Bolgatanga,Bolgatanga Municipality,Upper East,Ghana
706,Image_707,11.47,Non-Anemic,48,Female,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
707,Image_708,11.60,Non-Anemic,60,Male,Non-anemic,Komfo Anokye Teaching Hospital,Kumasi,Kumasi Metropolitan,Ashanti,Ghana
708,Image_709,12.10,Non-Anemic,48,Male,Non-anemic,Bolgatanga Regional Hospital,Bolgatanga,Bolgatanga Municipality,Upper East,Ghana


In [4]:
# Mapping diagnosis to severity
severity_mapping = {
    "Non-Anemic": 0,
    "Mild": 1,
    "Moderate": 2,
    "Severe": 3,
}

data_sheet['Severity'] = data_sheet['Severity'].map(severity_mapping)
display(data_sheet)

,IMAGE_ID,HB_LEVEL,Severity,Age(Months),GENDER,REMARK,HOSPITAL,CITY/TOWN,MUNICIPALITY/DISTRICT,REGION,COUNTRY
0,Image_001,9.80,2,6,Female,Anemic,Nkawie-Toase Government Hospital,Nkawie-Toase,Atwima Nwabiagya South,Ashanti,Ghana
1,Image_002,9.90,2,24,Male,Anemic,Ejusu Government Hospital,Ejusu,Ejusu Municipality,Ashanti,Ghana
2,Image_003,11.10,0,24,Female,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
3,Image_004,12.50,0,12,Male,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
4,Image_005,9.90,2,24,Male,Anemic,Sunyani Municipal Hospital,Sunyani,Sunyani Municipality,Bono,Ghana
...,...,...,...,...,...,...,...,...,...,...,...
705,Image_706,12.80,0,48,Male,Non-anemic,Bolgatanga Regional Hospital,Bolgatanga,Bolgatanga Municipality,Upper East,Ghana
706,Image_707,11.47,0,48,Female,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
707,Image_708,11.60,0,60,Male,Non-anemic,Komfo Anokye Teaching Hospital,Kumasi,Kumasi Metropolitan,Ashanti,Ghana
708,Image_709,12.10,0,48,Male,Non-anemic,Bolgatanga Regional Hospital,Bolgatanga,Bolgatanga Municipality,Upper East,Ghana


In [5]:
# Define data augmentations or transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=np.random.rand()),
    transforms.RandomVerticalFlip(p=np.random.rand()),
    transforms.RandomRotation(degrees=np.random.randint(0, 360)),
    transforms.RandomAffine(degrees=np.random.randint(0, 360)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Custom dataset class
class CPAnemiCDataset(Dataset):
    def __init__(self, dir, df, transform=None):
        self.dir = dir
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row['IMAGE_ID']
        img_folder = row['REMARK']
        img_path = os.path.join(self.dir, img_folder, img_id + ".png")
        img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        multiclass_label = torch.tensor(row['Severity'])
        hb_level = torch.tensor(row['HB_LEVEL'])

        return img, multiclass_label, hb_level

    # Load the dataset
image_dataset = CPAnemiCDataset(data_dir, data_sheet, transform=transform)
train_dataset, test_dataset = train_test_split(image_dataset, test_size=0.20, shuffle=True)

print(f"Image Dataset Size (All): {len(image_dataset)}, \
        Train Size: {len(train_dataset)}, \
        Test Size: {len(test_dataset)}")

BATCH_SIZE = 32
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

Image Dataset Size (All): 710,         Train Size: 568,         Test Size: 142


In [6]:
# Default device
device = torch.device('cpu')

# Check for CUDA availability
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    print("CUDA is not available, using CPU.")

print(f"Selected device: {device}")

Selected device: cuda


In [7]:
!nvidia-smi

Tue Feb 25 09:28:27 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.120                Driver Version: 550.120        CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:41:00.0 Off |                  Off |
|  0%   41C    P8             40W /  480W |     786MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [22]:
def get_model_size(mdl):
    torch.save(mdl.state_dict(), "tmp.pt")
    model_size = "Model Size: %.2f MB" %(os.path.getsize("tmp.pt")/1e6)
    os.remove('tmp.pt')
    return model_size

def timed_forward(model, img):
    """Applies checkpointing and logs GPU latency and memory usage."""
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    # Record memory before forward pass
    torch.cuda.reset_peak_memory_stats()
    mem_before = torch.cuda.memory_allocated()
    max_mem_before = torch.cuda.max_memory_allocated()

    # Start measuring latency
    start_event.record()
    class_pred, reg_pred = model(img)
    end_event.record()

    torch.cuda.synchronize()  # Ensure timing accuracy
    latency = start_event.elapsed_time(end_event)  # Time in milliseconds

    # Record memory after forward pass
    mem_after = torch.cuda.memory_allocated()
    max_mem_after = torch.cuda.max_memory_allocated()

    # Store stats in a dictionary
    stats = {
        "latency": latency,
        "malloc_before": mem_before,
        "malloc_after": mem_after,
        "max_malloc": max_mem_after,
    }

    return class_pred, reg_pred, stats

# Static Weighting Function. Set eta_class to desired importance (Classification > .5, Regression < .5, Equal == .5)
def sw_loss(loss_class, loss_reg, eta_class=0.5):
    eta_reg = 1 - eta_class
    total_loss = (eta_class * loss_class) + (eta_reg * loss_reg)
    return total_loss

def knowledge_distillation_loss(student_outputs, teacher_outputs, true_labels, true_levels, tau=7.0, alpha=0.90,epsilon=0.7):
    """
    Computes Knowledge Distillation loss.
    
    student_outputs: Tuple (student_class_logits, student_regression_output)
    teacher_outputs: Tuple (teacher_class_logits, teacher_regression_output)
    true_labels: Ground-truth labels for classification
    temperature: Softmax temperature scaling
    alpha: Weight for KD loss vs. hard label loss
    """
    student_logits, student_regression = student_outputs
    teacher_logits, teacher_regression = teacher_outputs

    # Compute soft targets from teacher using temperature scaling
    teacher_probs = F.softmax(teacher_logits / tau, dim=1)
    student_log_probs = F.log_softmax(student_logits / tau, dim=1)

    # KL Divergence loss between teacher and student soft labels
    kd_loss = nn.KLDivLoss(reduction='batchmean')(student_log_probs, teacher_probs)

    # Standard Cross-Entropy Loss with ground truth labels
    ce_loss = nn.CrossEntropyLoss()(student_logits, true_labels)

    # Mean Squared Error for Regression Loss
    mse_loss = nn.MSELoss()(student_regression.squeeze(), true_levels)

    # Weighted sum of losses
    return (alpha * kd_loss) + ((1 - alpha) * ((ce_loss*epsilon) + (mse_loss*(1-epsilon))))

In [23]:
class MultiModel(nn.Module):
    MODEL_MAPPING = {
        "mobilenetv2": lambda: models.mobilenet_v2(pretrained=False),
        "resnet18": lambda: models.resnet18(pretrained=False),
        "densenet121": lambda: models.densenet121(pretrained=False),
        "vgg16": lambda: models.vgg16(pretrained=False),
        "vit-tiny": lambda: create_model("vit_tiny_patch16_224", pretrained=False),
        "convnext-tiny": lambda: models.convnext_tiny(pretrained=False),
        "efficientnet-b0": lambda: models.efficientnet_b0(pretrained=False),
        "shufflenetv2-0.5x": lambda: models.shufflenet_v2_x0_5(pretrained=False),
        "regnety-400mf": lambda: models.regnet_y_400mf(pretrained=False),
        "mnasnet0_5": lambda: models.mnasnet0_5(pretrained=False),
        "ghostnetv2": lambda: create_model('ghostnetv2_100.in1k', pretrained=False),
        "tinynet-a": lambda: create_model("tinynet_a.in1k", pretrained=False)
    }

    FEATURE_LAYER_MAPPING = {
        "fc": ["resnet", "shufflenet", "regnet"],
        "classifier": ["densenet", "vgg", "mobilenet", "efficientnet",
                       "mnasnet","convnext", "ghostnet", "tinynet"],
        "head": ["vit"]
    }

    def __init__(self, model_name):
        super().__init__()
        self.model_name = model_name.lower()

        if self.model_name not in self.MODEL_MAPPING:
            raise ValueError(f"Model {model_name} not supported")

        self.model = self.MODEL_MAPPING[self.model_name]()
        num_ftrs = self._get_feature_size()

        print(f"Initial Backbone {get_model_size(self.model)}")

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.2),
            nn.Linear(num_ftrs, 128),
            nn.ReLU(),
            nn.Linear(128, 5)
        )

        self._assign_classifier()
        print(f"Modified Backbone {get_model_size(self.model)}\n")

    def _get_feature_size(self):
        """Retrieve the number of input features for the last layer."""
        # Special case for VGG16 since its features need flattening
        if "vgg" in self.model_name:
            return 25088  # VGG16 outputs (batch, 512, 7, 7) -> flattened to 25088

        feature_layers = {
            "fc": getattr(self.model, "fc", None),
            "classifier": getattr(self.model, "classifier", None),
            "head": getattr(self.model, "head", None)
        }

        for key, layer in feature_layers.items():
            if layer:
                return layer[-1].in_features if isinstance(layer, nn.Sequential) else layer.in_features

        return getattr(self.model, "num_features", None)

    def _assign_classifier(self):
        """Assigns the appropriate classifier to the model based on its architecture."""
        if "vgg" in self.model_name:
            self.model.classifier = self.classifier
        else:
          for attr, models in self.FEATURE_LAYER_MAPPING.items():
            if any(m in self.model_name for m in models):
                setattr(self.model, attr, self.classifier)
                return

    def forward(self, x):
        output = self.model(x)
        return output[:, :4], output[:, 4]  # Class probabilities and Hb level estimate

In [24]:
teacher_list = ["resnet18", "densenet121", "vgg16"]
student_list = ["mobilenetv2", "regnety-400mf", "shufflenetv2-0.5x"]

for arch in teacher_list:
    print(f"Loading model: {arch}")
    model = MultiModel(arch).to(device)

for arch in student_list:
    print(f"Loading model: {arch}")
    model = MultiModel(arch).to(device)

Loading model: resnet18
Initial Backbone Model Size: 46.83 MB
Modified Backbone Model Size: 45.04 MB

Loading model: densenet121
Initial Backbone Model Size: 32.47 MB
Modified Backbone Model Size: 28.90 MB

Loading model: vgg16
Initial Backbone Model Size: 553.44 MB
Modified Backbone Model Size: 71.72 MB

Loading model: mobilenetv2
Initial Backbone Model Size: 14.24 MB
Modified Backbone Model Size: 9.78 MB

Loading model: regnety-400mf
Initial Backbone Model Size: 17.61 MB
Modified Backbone Model Size: 16.07 MB

Loading model: shufflenetv2-0.5x
Initial Backbone Model Size: 5.59 MB
Modified Backbone Model Size: 2.02 MB



In [25]:
def train(dataloader, model, class_loss, reg1_loss, reg2_loss, optimizer):
    """Trains the model and logs additional metrics."""
    model.train()
    total_loss = 0
    total_ce_loss = 0
    total_mse_loss = 0
    total_mae_loss = 0
    correct = 0
    total_samples = 0

    all_preds = []
    all_targets = []
    all_probs = []
    all_hb_targets = []
    all_hb_preds = []

    for _, (img, multiclass, hb_level) in enumerate(dataloader):
        img = img.to(device)
        multiclass = multiclass.to(device).long()
        hb_level = hb_level.to(device).unsqueeze(1).float()

        optimizer.zero_grad()

        # Forward pass
        class_pred, reg_pred = model(img)

        with torch.no_grad():
            teacher_outputs = model(img)

        student_outputs = model(img)

        loss = knowledge_distillation_loss(student_outputs, teacher_outputs, multiclass, hb_level)

        # Compute losses
        ce_loss = class_loss(student_outputs[0], multiclass)
        mse_loss = reg1_loss(student_outputs[1], hb_level)
        mae_loss = reg2_loss(student_outputs[1], hb_level)

        # Backpropagation
        loss.backward()
        optimizer.step()

        # Track total losses
        total_loss += loss.item()
        total_ce_loss += ce_loss.item()
        total_mse_loss += mse_loss.item()
        total_mae_loss += mae_loss.item()

        # Compute classification accuracy
        class_probs = F.softmax(class_pred, dim=1)
        highest_prob_class = torch.argmax(class_probs, dim=1)

        correct += (highest_prob_class == multiclass).sum().item()
        total_samples += multiclass.size(0)

        # Collect data for additional metrics
        all_preds.extend(highest_prob_class.detach().cpu().numpy())
        all_targets.extend(multiclass.detach().cpu().numpy())
        all_probs.extend(class_probs.detach().cpu().numpy())
        all_hb_targets.extend(hb_level.detach().cpu().numpy())
        all_hb_preds.extend(reg_pred.squeeze().cpu().detach().numpy())

    # Compute additional metrics
    precision = precision_score(all_targets, all_preds, average="weighted")
    recall = recall_score(all_targets, all_preds, average="weighted")
    f1 = f1_score(all_targets, all_preds, average="weighted")
    auc = roc_auc_score(all_targets, all_probs, multi_class="ovr")
    r2 = r2_score(all_hb_targets, all_hb_preds)

    # Compute final statistics
    avg_loss = total_loss / len(dataloader)
    avg_ce_loss = total_ce_loss / len(dataloader)
    avg_mse_loss = total_mse_loss / len(dataloader)
    avg_mae_loss = total_mae_loss / len(dataloader)
    accuracy = correct / total_samples

    # Store metrics
    final_metrics = [avg_loss, avg_ce_loss, accuracy, precision, recall, f1, auc, r2, avg_mae_loss, avg_mse_loss]

    return final_metrics


In [26]:
def eval(dataloader, model, class_loss, reg1_loss, reg2_loss):
    """Evaluates the model with additional metrics: Precision, Recall, AUC, F1, R², Memory Usage, and Latency."""
    model.eval()
    mean_stats = []

    total_loss = 0
    total_ce_loss = 0
    total_mse_loss = 0
    total_mae_loss = 0
    correct = 0
    total_samples = 0

    all_preds = []
    all_targets = []
    all_probs = []
    all_hb_targets = []
    all_hb_preds = []

    torch.cuda.empty_cache()
    gc.collect()

    with torch.no_grad():
        for _, (img, multiclass, hb_level) in enumerate(dataloader):
            img = img.to(device)
            multiclass = multiclass.to(device).long()
            hb_level = hb_level.to(device).unsqueeze(1).float()

            # Forward pass with latency & memory tracking
            class_pred, reg_pred, stats = timed_forward(model, img)
            mean_stats.append(stats)

            # Compute losses
            ce_loss = class_loss(class_pred, multiclass)
            mse_loss = reg1_loss(reg_pred, hb_level)
            mae_loss = reg2_loss(reg_pred, hb_level)
            loss = sw_loss(ce_loss, mse_loss, 0.7)

            # Track total losses
            total_loss += loss.item()
            total_ce_loss += ce_loss.item()
            total_mse_loss += mse_loss.item()
            total_mae_loss += mae_loss.item()

            # Compute classification accuracy
            class_probs = F.softmax(class_pred, dim=1)
            highest_prob_class = torch.argmax(class_probs, dim=1)

            correct += (highest_prob_class == multiclass).sum().item()
            total_samples += multiclass.size(0)

            # Collect data for additional metrics
            all_preds.extend(highest_prob_class.detach().cpu().numpy())
            all_targets.extend(multiclass.detach().cpu().numpy())
            all_probs.extend(class_probs.detach().cpu().numpy())
            all_hb_targets.extend(hb_level.detach().cpu().numpy())
            all_hb_preds.extend(reg_pred.squeeze().detach().cpu().numpy())

    # Compute mean statistics
    mean_latency = np.mean([s["latency"] for s in mean_stats])
    mean_mem_before = np.mean([s["malloc_before"] for s in mean_stats]) / 1_048_576  # Convert bytes to MB
    mean_mem_after = np.mean([s["malloc_after"] for s in mean_stats]) / 1_048_576  # Convert bytes to MB
    mean_max_mem = np.mean([s["max_malloc"] for s in mean_stats]) / 1_048_576  # Convert bytes to MB

    # Store final mean statistics
    final_mean_stats = [mean_latency, mean_mem_before, mean_mem_after, mean_max_mem]

    # Compute additional evaluation metrics
    precision = precision_score(all_targets, all_preds, average="weighted")
    recall = recall_score(all_targets, all_preds, average="weighted")
    f1 = f1_score(all_targets, all_preds, average="weighted")
    auc = roc_auc_score(all_targets, all_probs, multi_class="ovr")
    r2 = r2_score(all_hb_targets, all_hb_preds)

    # Compute confusion matrix
    # cm = confusion_matrix(all_targets, all_preds)

    # Compute final average losses
    avg_loss = total_loss / len(dataloader)
    avg_ce_loss = total_ce_loss / len(dataloader)
    avg_mse_loss = total_mse_loss / len(dataloader)
    avg_mae_loss = total_mae_loss / len(dataloader)
    accuracy = correct / total_samples

    # Store metrics
    final_metrics = [avg_loss, avg_ce_loss, accuracy, precision, recall, f1, auc, r2, avg_mae_loss, avg_mse_loss]

    return final_metrics, final_mean_stats

In [27]:
def main():
    # Device setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Define loss functions
    cross_entropy_loss = torch.nn.CrossEntropyLoss()  # Multi-class classification loss
    mse_loss = torch.nn.MSELoss()  # Regression loss
    mae_loss = torch.nn.L1Loss()  # Regression loss

    # Set up 5-Fold Cross Validation
    kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

    # Model Saving Directory
    weights_dir = "/home/sebastian-cruz6/cp-anemia-detection/notebooks/weights"
    os.makedirs(weights_dir, exist_ok=True)

    metrics_dir = "/home/sebastian-cruz6/cp-anemia-detection/notebooks/metrics"
    os.makedirs(metrics_dir, exist_ok=True)

    print("=" * 100)
    print(f"Teacher Model: {TEACHER_ARCH}")
    print(f"Student Model: {STUDENT_ARCH}")

    # === INITIALIZE MODEL ===
    teacher_model = MultiModel(TEACHER_ARCH).to(device)
    teacher_model.load_state_dict(torch.load(f"{weights_dir}/pytorch/model_best_accuracy_{TEACHER_ARCH}_02082024.pth"))

    student_model = MultiModel(STUDENT_ARCH).to(device)

    optimizer = torch.optim.Adam(student_model.parameters(), lr=1e-4)

    best_val_acc = -float("inf")  # Track best validation accuracy
    train_metrics_list = []
    val_metrics_list = []

    # === TRAINING LOOP ===
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        fold = 1

        for train_idx, val_idx in kf.split(range(len(image_dataset))):
            train_subset = Subset(image_dataset, train_idx)
            val_subset = Subset(image_dataset, val_idx)

            train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
            val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

            if fold == FOLDS:
                # === VALIDATION PHASE ===
                val_metrics, val_stats = eval(val_loader, student_model, cross_entropy_loss, mse_loss, mae_loss)
                print(
                    f"Validation: Fold {fold} - Total Loss: {val_metrics[0]:.4f}, Cross Entropy: {val_metrics[1]:4f}, Accuracy: {val_metrics[2]:.4f}, "
                    f"Precision: {val_metrics[3]:.4f}, Recall: {val_metrics[4]:.4f}, F1 Score: {val_metrics[5]:.4f}, AUC: {val_metrics[6]:.4f}, "
                    f"R2 Score: {val_metrics[7]:4f}, MAE: {val_metrics[8]:.4f}, MSE: {val_metrics[9]:.4f}"
                )
                print(
                    f"Avg Latency (ms): {val_stats[0]:.2f}, Avg Memory Before (MB): {val_stats[1]:.2f}, "
                    f"Avg Memory After (MB): {val_stats[2]:.2f}, Avg Max Memory (MB): {val_stats[3]:.2f}"
                )

                # Save best model based on validation accuracy
                if val_metrics[2] > best_val_acc:
                    best_val_acc = val_metrics[2]
                    torch.save(
                        model.state_dict(),
                        f"{weights_dir}/pytorch/model_best_accuracy_student{STUDENT_ARCH}_{signature}.pth",
                    )
                    print(f"Best model saved with Accuracy: {best_val_acc:.4f}")

                # Store validation metrics
                val_metrics_list.append(
                    {
                        "epoch": epoch + 1,
                        "fold": fold,
                        "total_loss": val_metrics[0],
                        "cross_entropy_loss": val_metrics[1],
                        "accuracy": val_metrics[2],
                        "precision": val_metrics[3],
                        "recall": val_metrics[4],
                        "f1_score": val_metrics[5],
                        "auc": val_metrics[6],
                        "r2_score": val_metrics[7],
                        "mae_loss": val_metrics[8],
                        "mse_loss": val_metrics[9],
                        "latency": val_stats[0],
                        "malloc_before": val_stats[1],
                        "malloc_after": val_stats[2],
                        "max_malloc": val_stats[3],
                    }
                )

            else:
                # === TRAINING PHASE ===
                train_metrics = train(train_loader, student_model, cross_entropy_loss, mse_loss, mae_loss, optimizer)
                print(
                    f"Training: Fold {fold} - Total Loss: {train_metrics[0]:.4f}, Cross Entropy: {train_metrics[1]:4f}, Accuracy: {train_metrics[2]:.4f}, "
                    f"Precision: {train_metrics[3]:.4f}, Recall: {train_metrics[4]:.4f}, F1 Score: {train_metrics[5]:.4f}, AUC: {train_metrics[6]:.4f}, "
                    f"R2 Score: {train_metrics[7]:4f}, MAE: {train_metrics[8]:.4f}, MSE: {train_metrics[9]:.4f}"
                )

                # Store training metrics
                train_metrics_list.append(
                    {
                        "epoch": epoch + 1,
                        "fold": fold,
                        "total_loss": train_metrics[0],
                        "cross_entropy_loss": train_metrics[1],
                        "accuracy": train_metrics[2],
                        "precision": train_metrics[3],
                        "recall": train_metrics[4],
                        "f1_score": train_metrics[5],
                        "auc": train_metrics[6],
                        "r2_score": train_metrics[7],
                        "mae_loss": train_metrics[8],
                        "mse_loss": train_metrics[9],
                    }
                )

            fold += 1  # Move to next fold
        
        keys = train_metrics_list[0].keys()
        with open(f"{metrics_dir}/pytorch/training_metrics_student{STUDENT_ARCH}_{signature}.csv", 'w', newline='') as output_file:
            dict_writer = csv.DictWriter(output_file, keys)
            dict_writer.writeheader()
            dict_writer.writerows(train_metrics_list)

        keys = val_metrics_list[0].keys()
        with open(f"{metrics_dir}/pytorch/validation_metrics_student{STUDENT_ARCH}_{signature}.csv", 'w', newline='') as output_file:
            dict_writer = csv.DictWriter(output_file, keys)
            dict_writer.writeheader()
            dict_writer.writerows(val_metrics_list)

    print(f"\nFine-tuned {get_model_size(model)}")
    print("=" * 100)

In [28]:
# === CONFIGURATION ===
TEACHER_ARCH = "vgg16"
STUDENT_ARCH = "shufflenetv2-0.5x"
BATCH_SIZE = 32
EPOCHS = 150
FOLDS = 5

main()

Teacher Model: vgg16
Student Model: shufflenetv2-0.5x
Initial Backbone Model Size: 553.44 MB
Modified Backbone Model Size: 71.72 MB

Initial Backbone Model Size: 5.59 MB
Modified Backbone Model Size: 2.02 MB


Epoch 1/150
Training: Fold 1 - Total Loss: 2.8885, Cross Entropy: 1.454756, Accuracy: 0.0810, Precision: 0.2309, Recall: 0.0810, F1 Score: 0.0383, AUC: 0.4906, R2 Score: -16.818007, MAE: 9.3512, MSE: 92.8894
Training: Fold 2 - Total Loss: 1.9145, Cross Entropy: 1.485357, Accuracy: 0.0634, Precision: 0.0040, Recall: 0.0634, F1 Score: 0.0076, AUC: 0.5392, R2 Score: -11.195718, MAE: 7.4148, MSE: 60.3503
Training: Fold 3 - Total Loss: 1.1192, Cross Entropy: 1.474610, Accuracy: 0.2641, Precision: 0.1487, Recall: 0.2641, F1 Score: 0.1868, AUC: 0.5234, R2 Score: -5.953120, MAE: 5.3863, MSE: 33.8664
Training: Fold 4 - Total Loss: 0.6339, Cross Entropy: 1.365565, Accuracy: 0.4173, Precision: 0.1741, Recall: 0.4173, F1 Score: 0.2457, AUC: 0.5000, R2 Score: -2.370389, MAE: 3.7476, MSE: 17.9

In [ ]:
# === CONFIGURATION ===
TEACHER_ARCH = "vgg16"
STUDENT_ARCH = "mobilenetv2"
BATCH_SIZE = 32
EPOCHS = 150
FOLDS = 5

main()

In [ ]:
# === CONFIGURATION ===
TEACHER_ARCH = "vgg16"
STUDENT_ARCH = "regnety-400mf"
BATCH_SIZE = 32
EPOCHS = 150
FOLDS = 5

main()

In [ ]:
# === CONFIGURATION ===
TEACHER_ARCH = "resnet18"
STUDENT_ARCH = "shufflenetv2-0.5x"
BATCH_SIZE = 32
EPOCHS = 150
FOLDS = 5

main()

In [ ]:
# === CONFIGURATION ===
TEACHER_ARCH = "resnet18"
STUDENT_ARCH = "mobilenetv2"
BATCH_SIZE = 32
EPOCHS = 150
FOLDS = 5

In [ ]:
# === CONFIGURATION ===
TEACHER_ARCH = "resnet18"
STUDENT_ARCH = "regnety-400mf"
BATCH_SIZE = 32
EPOCHS = 150
FOLDS = 5

In [ ]:
# === CONFIGURATION ===
TEACHER_ARCH = "densenet121"
STUDENT_ARCH = "shufflenetv2-0.5x"
BATCH_SIZE = 32
EPOCHS = 150
FOLDS = 5

main()

In [ ]:
# === CONFIGURATION ===
TEACHER_ARCH = "densenet121"
STUDENT_ARCH = "mobilenetv2"
BATCH_SIZE = 32
EPOCHS = 150
FOLDS = 5

main()

In [ ]:
# === CONFIGURATION ===
TEACHER_ARCH = "densenet121"
STUDENT_ARCH = "regnety-400mf"
BATCH_SIZE = 32
EPOCHS = 150
FOLDS = 5

main()

## Inference

In [33]:
for arch in teacher_list:
    cross_entropy_loss = torch.nn.CrossEntropyLoss()  # Multi-class classification loss
    mse_loss = torch.nn.MSELoss()  # Regression loss
    mae_loss = torch.nn.L1Loss()  # Regression loss

    test_metrics_list = []
    print("="*100)
    print(f"{arch}")
    model = MultiModel(arch).to(device)
    model.load_state_dict(torch.load(f"{weights_dir}pytorch/model_best_accuracy_{arch}_02082024.pth"))

    # === Testing PHASE ===
    test_metrics, test_stats = eval(test_loader, model, cross_entropy_loss, mse_loss, mae_loss)
    print(
        f"Testing: Total Loss: {test_metrics[0]:.4f}, Cross Entropy: {test_metrics[1]:4f}, Accuracy: {test_metrics[2]:.4f}, "
        f"Precision: {test_metrics[3]:.4f}, Recall: {test_metrics[4]:.4f}, F1 Score: {test_metrics[5]:.4f}, AUC: {test_metrics[6]:.4f}, "
        f"R2 Score: {test_metrics[7]:4f}, MAE: {test_metrics[8]:.4f}, MSE: {test_metrics[9]:.4f}"
    )
    print(
        f"Avg Latency (ms): {test_stats[0]:.2f}, Avg Memory Before (MB): {test_stats[1]:.2f}, "
        f"Avg Memory After (MB): {test_stats[2]:.2f}, Avg Max Memory (MB): {test_stats[3]:.2f}"
    )

    # Store validation metrics
    test_metrics_list.append(
        {
            "total_loss": test_metrics[0],
            "cross_entropy_loss": test_metrics[1],
            "accuracy": test_metrics[2],
            "precision": test_metrics[3],
            "recall": test_metrics[4],
            "f1_score": test_metrics[5],
            "auc": test_metrics[6],
            "r2_score": test_metrics[7],
            "mae_loss": test_metrics[8],
            "mse_loss": test_metrics[9],
            "latency": test_stats[0],
            "malloc_before": test_stats[1],
            "malloc_after": test_stats[2],
            "max_malloc": test_stats[3],
        }
    )

    keys = test_metrics_list[0].keys()
    with open(f"{metrics_dir}/pytorch/testing_metrics_{arch}_{signature}.csv", 'w', newline='') as output_file:
        dict_writer = csv.DictWriter(output_file, keys)
        dict_writer.writeheader()
        dict_writer.writerows(test_metrics_list)

resnet18
Initial Backbone Model Size: 46.83 MB
Modified Backbone Model Size: 45.04 MB

Testing: Total Loss: 2.3230, Cross Entropy: 0.384034, Accuracy: 0.8239, Precision: 0.8384, Recall: 0.8239, F1 Score: 0.8233, AUC: 0.9648, R2 Score: -0.051626, MAE: 2.0297, MSE: 6.8471
Avg Latency (ms): 6.16, Avg Memory Before (MB): 77.90, Avg Memory After (MB): 77.90, Avg Max Memory (MB): 251.85
densenet121
Initial Backbone Model Size: 32.47 MB
Modified Backbone Model Size: 28.90 MB

Testing: Total Loss: 2.2034, Cross Entropy: 0.300885, Accuracy: 0.8662, Precision: 0.8762, Recall: 0.8662, F1 Score: 0.8642, AUC: 0.9762, R2 Score: 0.011895, MAE: 2.0020, MSE: 6.6425
Avg Latency (ms): 23.48, Avg Memory Before (MB): 60.65, Avg Memory After (MB): 60.65, Avg Max Memory (MB): 332.95
vgg16
Initial Backbone Model Size: 553.44 MB
Modified Backbone Model Size: 71.72 MB

Testing: Total Loss: 2.2334, Cross Entropy: 0.370047, Accuracy: 0.8451, Precision: 0.8731, Recall: 0.8451, F1 Score: 0.8423, AUC: 0.9706, R2 Sco

In [34]:
for arch in student_list:
    cross_entropy_loss = torch.nn.CrossEntropyLoss()  # Multi-class classification loss
    mse_loss = torch.nn.MSELoss()  # Regression loss
    mae_loss = torch.nn.L1Loss()  # Regression loss

    test_metrics_list = []
    print("="*100)
    print(f"{arch}")
    model = MultiModel(arch).to(device)
    model.load_state_dict(torch.load(f"{weights_dir}/pytorch/model_best_accuracy_student{arch}_02082024.pth"))

    # === Testing PHASE ===
    test_metrics, test_stats = eval(test_loader, model, cross_entropy_loss, mse_loss, mae_loss)
    print(
        f"Testing: Total Loss: {test_metrics[0]:.4f}, Cross Entropy: {test_metrics[1]:4f}, Accuracy: {test_metrics[2]:.4f}, "
        f"Precision: {test_metrics[3]:.4f}, Recall: {test_metrics[4]:.4f}, F1 Score: {test_metrics[5]:.4f}, AUC: {test_metrics[6]:.4f}, "
        f"R2 Score: {test_metrics[7]:4f}, MAE: {test_metrics[8]:.4f}, MSE: {test_metrics[9]:.4f}"
    )
    print(
        f"Avg Latency (ms): {test_stats[0]:.2f}, Avg Memory Before (MB): {test_stats[1]:.2f}, "
        f"Avg Memory After (MB): {test_stats[2]:.2f}, Avg Max Memory (MB): {test_stats[3]:.2f}"
    )

    # Store validation metrics
    test_metrics_list.append(
        {
            "total_loss": test_metrics[0],
            "cross_entropy_loss": test_metrics[1],
            "accuracy": test_metrics[2],
            "precision": test_metrics[3],
            "recall": test_metrics[4],
            "f1_score": test_metrics[5],
            "auc": test_metrics[6],
            "r2_score": test_metrics[7],
            "mae_loss": test_metrics[8],
            "mse_loss": test_metrics[9],
            "latency": test_stats[0],
            "malloc_before": test_stats[1],
            "malloc_after": test_stats[2],
            "max_malloc": test_stats[3],
        }
    )

    keys = test_metrics_list[0].keys()
    with open(f"{metrics_dir}/pytorch/testing_metrics_{arch}_{signature}.csv", 'w', newline='') as output_file:
        dict_writer = csv.DictWriter(output_file, keys)
        dict_writer.writeheader()
        dict_writer.writerows(test_metrics_list)

mobilenetv2
Initial Backbone Model Size: 14.24 MB
Modified Backbone Model Size: 9.78 MB

Testing: Total Loss: 2.2938, Cross Entropy: 0.420071, Accuracy: 0.8169, Precision: 0.8260, Recall: 0.8169, F1 Score: 0.8145, AUC: 0.9558, R2 Score: -0.067216, MAE: 2.0048, MSE: 6.6658
Avg Latency (ms): 7.53, Avg Memory Before (MB): 42.90, Avg Memory After (MB): 42.91, Avg Max Memory (MB): 327.97
regnety-400mf
Initial Backbone Model Size: 17.61 MB
Modified Backbone Model Size: 16.07 MB

Testing: Total Loss: 2.2945, Cross Entropy: 0.403165, Accuracy: 0.8028, Precision: 0.8281, Recall: 0.8028, F1 Score: 0.7973, AUC: 0.9549, R2 Score: -0.042162, MAE: 2.0105, MSE: 6.7075
Avg Latency (ms): 4.03, Avg Memory Before (MB): 47.87, Avg Memory After (MB): 47.87, Avg Max Memory (MB): 239.73
shufflenetv2-0.5x
Initial Backbone Model Size: 5.59 MB
Modified Backbone Model Size: 2.02 MB

Testing: Total Loss: 2.3940, Cross Entropy: 0.537119, Accuracy: 0.7394, Precision: 0.7597, Recall: 0.7394, F1 Score: 0.7349, AUC: 0